# Notebook for the Complete Framework of elfe3D_GPR Simulations:

1. Creating input files for one simulation using io's input toolbox.
2. Running tetgen to create mesh files for the input problem.
3. Running elfe3D_GPR executable to simulate the input problem.
4. Extracting the result and post-processing and visualizing it using io's output toolbox.

# Step 0: All the Paths

In [1]:
# Change working directory to project root for consistent relative paths
import os
from pathlib import Path

io_dir = Path(os.curdir).parent.resolve()
inputs_dir = io_dir / "inputs"
outputs_dir = io_dir / "outputs" 
exec_dir = Path(os.curdir).parent.parent.resolve()
exec_inputs_dir = exec_dir / "in"

print(f"Current working directory: {os.getcwd()}")
print(f"io_dir: {io_dir}")
print(f"inputs_dir: {inputs_dir}")
print(f"outputs_dir: {outputs_dir}")
print(f"exec_dir: {exec_dir}")
print(f"exec_inputs_dir: {exec_inputs_dir}")

Current working directory: f:\Projects\EMGeoInversion\elfe3D_GPR\io
io_dir: F:\Projects\EMGeoInversion\elfe3D_GPR\io
inputs_dir: F:\Projects\EMGeoInversion\elfe3D_GPR\io\inputs
outputs_dir: F:\Projects\EMGeoInversion\elfe3D_GPR\io\outputs
exec_dir: F:\Projects\EMGeoInversion\elfe3D_GPR\io
exec_inputs_dir: F:\Projects\EMGeoInversion\elfe3D_GPR\io\in


## Step 1. Creating Input Files

In [2]:
"""
example_runs.py
---------------
Usage examples for GPRSurvey showing all combinations:

  Case 1 — Air only, no anomaly       (num_layers=0, anomaly=None)
  Case 2 — Layered earth, no anomaly  (num_layers=2, anomaly=None)
  Case 3 — Layered earth + anomaly    (num_layers=2, anomaly=BoxAnomaly)

"""

# Changing to the inputs directory for the input files modules
os.chdir(inputs_dir)
# Now import the input survey class
from survey import GPRSurvey

# Shared frequency / wavelength setup

f = 100e6                   # 100 MHz central frequency
wave = 3e8 / f              # free-space wavelength in air = 3.0 m


# =============================================================================
# Case 1: Air only, no anomaly
# =============================================================================

survey_air = GPRSurvey.build(
    experiment_name="air_only",

    # Domain
    x_e=[-wave/10, 1 + wave/10],
    y_e=[-wave/10, wave/10],
    z_e=[-wave/10, wave/10],

    # Materials — air only, no earth layers
    air_eps_r=1.0,
    air_sigma=1e-16,
    layer_thicknesses=[wave/10], # creating this layer at the middle of the domain for ease of whole-space model PML setup. Unfortunately, the current implementation of the layered media requires at least one layer to be defined.
    layer_eps_r=[1.0], # same as air
    layer_sigma=[1e-16], # same as air
    layer_mu_r=[1.0], # same as air
    layer_sigma_m=[0.0], # same as air

    # No anomaly — omit anomaly_x/y/z entirely

    # Source
    ricker_central_f=f,
    num_points_per_range=1,
    antenna_position=[0.0, 0.0, 0.025],
    source_type=6,
    current_direction=1,
    num_segments=1,
    s_f=250,
    bh_f=1.0,
    box_present=False,
    box_x=[-1.0, 1.0],

    # Receivers
    num_receivers_inline=48,
    num_receivers_endfire=0,
    num_receivers_oblique=0,

    # Solver
    solver_type=2,
    max_ref_steps=0,
    max_unknowns=5_000_000,
    accuracy_tol=3e-5,
    output_fields_vtk=1,

    # PML
    num_pml_layers=1,
    pml_layer_thickness=wave/10,
    pml_type="lin",
    pml_decay_type=1,

    # Mesh sizing
    least_samples_per_wavelength=20,
)

survey_air.generate()


# =============================================================================
# Case 2: Two earth layers, no anomaly
# =============================================================================

survey_layered = GPRSurvey.build(
    experiment_name="layered_no_anomaly",

    # Domain
    x_e=[-wave/10, 1 + wave/10],
    y_e=[-wave/10, wave/10],
    z_e=[-1.0 - wave/10/3, wave/10],

    # Materials — air + 2 earth layers
    air_eps_r=1.0,
    air_sigma=1e-16,
    layer_thicknesses=[1.0, wave/10/3],
    layer_eps_r=[4.0, 9.0],
    layer_sigma=[1e-4, 1e-3],
    layer_mu_r=[1.0, 1.0],
    layer_sigma_m=[0.0, 0.0],

    # No anomaly — omit anomaly_x/y/z entirely

    # Source
    ricker_central_f=f,
    num_points_per_range=1,
    antenna_position=[0.0, 0.0, 0.025],
    source_type=6,
    current_direction=1,
    num_segments=1,
    s_f=250,
    bh_f=1.0,
    box_present=False,
    box_x=[-1 + 0.75, 1 + 0.375],

    # Receivers
    num_receivers_inline=48,
    num_receivers_endfire=0,
    num_receivers_oblique=0,

    # Solver
    solver_type=2,
    max_ref_steps=0,
    max_unknowns=5_000_000,
    accuracy_tol=3e-5,
    output_fields_vtk=1,

    # PML
    num_pml_layers=1,
    pml_layer_thickness=wave/10,
    pml_type="lin",
    pml_decay_type=1,

    least_samples_per_wavelength=20,
)

survey_layered.generate()


# ==================================
# Case 3: Two earth layers + anomaly
# ==================================

survey_anomaly = GPRSurvey.build(
    experiment_name="AnAir",

    # Domain
    x_e=[-wave/10, 1 + wave/10],
    y_e=[-wave/10, wave/10],
    z_e=[-1.0 - wave/10/3, wave/10],

    # Materials — air + 2 earth layers
    air_eps_r=1.0,
    air_sigma=1e-16,
    layer_thicknesses=[1.0, wave/10/3],
    layer_eps_r=[4.0, 9.0],
    layer_sigma=[1e-4, 1e-3],
    layer_mu_r=[1.0, 1.0],
    layer_sigma_m=[0.0, 0.0],

    # Anomaly — all three coordinate tuples + properties required together
    anomaly_x=(0, wave/8),
    anomaly_y=(-wave/20, wave/20),
    anomaly_z=(-0.9, -0.5),
    anomaly_properties=(20, 1e-4, 1.0, 0.0),   # (eps_r, sigma, mu_r, sigma_m)

    # Source
    ricker_central_f=f,
    num_points_per_range=1,
    antenna_position=[0.0, 0.0, 0.025],
    source_type=6,
    current_direction=1,
    num_segments=1,
    s_f=250,
    bh_f=1.0,
    box_present=False,
    box_x=[-1 + 0.75, 1 + 0.375],
    m=5,

    # Receivers
    num_receivers_inline=48,
    num_receivers_endfire=0,
    num_receivers_oblique=0,

    # Solver
    solver_type=2,
    max_ref_steps=0,
    max_unknowns=5_000_000,
    accuracy_tol=3e-5,
    output_fields_vtk=1,

    # PML
    num_pml_layers=1,
    pml_layer_thickness=wave/10,
    pml_type="lin",
    pml_decay_type=1,

    least_samples_per_wavelength=20,
)

survey_anomaly.generate()


odepths: [0.0375, 0.0375]
Source antenna length: 0.0006 m
Receiver antenna depth: -0.00015 m
[0.0]
Written: in_air_only\GPR_model_air_only.poly
Written: in_air_only\elfe3D_input.txt
Written: in_air_only\source.txt
Written: in_air_only\regionparameters.txt
Input generation complete.
odepths: [0.0375, 0.01875, 0.0125]
Source antenna length: 0.0003 m
Receiver antenna depth: -7.5e-05 m
[0.0, -1.0]
Written: in_layered_no_anomaly\GPR_model_layered_no_anomaly.poly
Written: in_layered_no_anomaly\elfe3D_input.txt
Written: in_layered_no_anomaly\source.txt
Written: in_layered_no_anomaly\regionparameters.txt
Input generation complete.
odepths: [0.0375, 0.01875, 0.0125]
Source antenna length: 0.0003 m
Receiver antenna depth: -7.5e-05 m
[0.0, -1.0]
Written: in_AnAir\GPR_model_AnAir.poly
Written: in_AnAir\elfe3D_input.txt
Written: in_AnAir\source.txt
Written: in_AnAir\regionparameters.txt
Input generation complete.


## Adjustable Paths and Parameters

### Define all adjustable paths here. These point to:
- workspace_root: The root of the project (F:\Projects\EMGeoInversion\elfe3D_GPR)
- io_dir: The io folder containing inputs and outputs subfolders
- inputs_dir: Where input files are generated (io/inputs)
- exe_dir: Directory containing the FEM executable (elfe3D_GPR)
- exe_name: Name of the executable (elfe3d_gpr)
- experiment_name: From the survey (e.g., "air_only") - adjust based on which case you run

In [3]:
import os

### Adjustable paths
workspace_root = r"F:\Projects\EMGeoInversion\elfe3D_GPR"  # Root of the elfe3D_GPR project
io_dir = os.path.join(workspace_root, "io")  # io folder
inputs_dir = os.path.join(io_dir, "inputs")  # inputs subfolder
exe_dir = os.path.join(workspace_root, "elfe3D_GPR")  # Directory with the executable
exe_name = "elfe3d_gpr"  # Executable name

### Experiment name - change this to match the survey you run (e.g., "air_only", "layered_no_anomaly", "AnAir")
experiment_name = "air_only"

### Derived paths
input_folder = os.path.join(inputs_dir, f"in_{experiment_name}")
poly_file = f"GPR_model_{experiment_name}.poly"
exe_path = os.path.join(exe_dir, exe_name)

### WSL paths (for running in WSL from Windows)
def to_wsl_path(win_path):
    return win_path.replace("F:", "/mnt/f").replace("\\", "/")

wsl_input_folder = to_wsl_path(input_folder)
wsl_exe_dir = to_wsl_path(exe_dir)

## Step 2: Calling tetgen now to generate mesh files

In [4]:
import subprocess

# Run tetgen in WSL to generate mesh files
# Tetgen creates mesh files in the same directory as the input .poly file
tetgen_command = f'cd {wsl_input_folder} && tetgen15 -pq1.2kAaen {poly_file}'
print(f"Running tetgen: {tetgen_command}")
get_ipython().system(f"wsl bash -c '{tetgen_command}'")

## Step 3: Calling elfe3D_GPR now to run the simulation

Running tetgen: cd /mnt/f/Projects/EMGeoInversion/elfe3D_GPR/io/inputs/in_air_only && tetgen15 -pq1.2kAaen GPR_model_air_only.poly


/bin/bash: -c: line 0: unexpected EOF while looking for matching `''
/bin/bash: -c: line 1: syntax error: unexpected end of file


## Step 3: Calling elfe3D_GPR now to run the simulation

In [ ]:
# Run the elfe3D_GPR executable in WSL
# The executable is run from its directory; it reads input files from relative paths or current dir
elfe_command = f"cd {wsl_exe_dir} && ./{exe_name}"
print(f"Running elfe3D_GPR: {elfe_command}")
result = subprocess.run(["wsl", "bash", "-c", elfe_command], capture_output=True, text=True)
if result.returncode != 0:
    print("elfe3D_GPR failed:")
    print(result.stderr)
else:
    print("elfe3D_GPR completed successfully.")
    print(result.stdout)

Running elfe3D_GPR: cd /mnt/f/Projects/EMGeoInversion/elfe3D_GPR/elfe3D_GPR && ./elfe3d_gpr


## Step 4: Post-processing results

In [ ]:
from fieldreader import AnalyticalLoader, ElfeLoader, load_elfe_batch
from postprocess import field_error, all_errors, error_stats
from visualize   import (
    ReceiverLinePlot,
    ReceiverLineErrorPlot,
    ReceiverLineCombined,
    ReceiverLineCombinedMulti,
    ErrorHistogramPlot,
    ErrorStatPlot,
)

# ---------------------------------------------------------------------------
# Paths  –  edit these
# ---------------------------------------------------------------------------

BASE        = r"F:\Projects\EMGeoInversion\Tests_Thesis"
ANALYTICAL  = os.path.join(BASE, "semi-analytic_100MHz")


# ===========================================================================
# Section 1 – Half-space CMP  (plotting_th2)
# ===========================================================================

POST = os.path.join(BASE, "6HS", "postprocess")

evert   = AnalyticalLoader(os.path.join(ANALYTICAL, "Exx_single_freq_4_100MHz.csv"),   label="Evert").endfire()
empymod = AnalyticalLoader(os.path.join(ANALYTICAL, "GPR-2001-4-dlf.csv"), label="empymod - 2001 DLF").endfire()

loaders = load_elfe_batch(
    base_folder=os.path.join(BASE, "6HS"),
    run_names=["out_HF_l1d_l2PML_CMP_BA", "out_HF_l1d_l2PML_CMP_BK", "out_HF_l1d_l2PML_CMP_F"],
    labels=[r"Uniform $k_\text{min}$", r"Uniform $k_\text{max}$", "Varying Stretch"],
    num_endfire=256,
)
runs = [l.endfire() for l in loaders]

ReceiverLinePlot([evert, empymod] + runs).plot(
    suptitle="Endfire – Half-Space CMP",
    output_dir=POST, fname="hs_cmp_comparison.png",
)
ReceiverLineErrorPlot([evert, empymod] + runs, reference=evert).plot(
    suptitle="Errors vs Evert – Half-Space CMP",
    output_dir=POST, fname="hs_cmp_error.png",
)
ReceiverLineCombinedMulti(runs, analytical=evert).plot(
    suptitle="Combined – Half-Space CMP",
    output_dir=POST, fname="hs_cmp_combined.png",
)


# # ===========================================================================
# # Section 2 – Two-layer CO  (plotting_th3)
# # ===========================================================================

# POST2 = os.path.join(BASE, "6TL", "postprocess")

# evert_tl = AnalyticalLoader(
#     os.path.join(ANALYTICAL, "Exx_single_freq_4_9_100MHz_NR.csv"), label="Evert"
# ).endfire()

# loaders2 = load_elfe_batch(
#     base_folder=os.path.join(BASE, "6TL"),
#     run_names=["out_TL_l1d_l2PML_CO_BA", "out_TL_l1d_l2PML_CO_F"],
#     labels=[r"Uniform Stretch – Air Like PML 1.5 m", r"Varying Stretch – Earth-Like PML 0.75 m"],
#     num_endfire=48,
# )
# runs2 = [l.endfire() for l in loaders2]

# ReceiverLinePlot([evert_tl] + runs2).plot(
#     suptitle="Endfire – Two-Layer CO",
#     output_dir=POST2, fname="tl_co_comparison.png",
# )
# ReceiverLineErrorPlot(runs2, reference=evert_tl).plot(
#     suptitle="Errors vs Evert – Two-Layer CO",
#     output_dir=POST2, fname="tl_co_error.png",
# )
# ErrorHistogramPlot(runs2, reference=evert_tl).plot(
#     suptitle="Error Histogram – Two-Layer CO",
#     output_dir=POST2, fname="tl_co_hist.png",
# )


# # ===========================================================================
# # Section 3 – PML thickness convergence  (plotting_thesis_final)
# # ===========================================================================

# POST3 = os.path.join(BASE, "Fin", "postprocess")

# evert_fin = AnalyticalLoader(
#     os.path.join(ANALYTICAL, "Exx_single_freq_4_9_100MHz_NR.csv"), label="Evert"
# ).endfire()

# wave    = 3.0   # wavelength in second layer (m)
# denoms  = [10, 12.5, 15, 17.5, 20, 22.5, 25]
# loaders3 = load_elfe_batch(
#     base_folder=os.path.join(BASE, "Fin"),
#     run_names=[f"run_pml_{str(d).replace('.', 'p')}" for d in denoms],
#     labels=[rf"$\lambda/{d}$" for d in denoms],
#     num_endfire=48,
# )
# runs3 = [l.endfire() for l in loaders3]

# xtick_labels = [rf"$\dfrac{{\lambda}}{{{d}}}$" for d in denoms]

# ErrorStatPlot(
#     runs3, reference=evert_fin,
#     param_values=[wave / d for d in denoms],
#     xtick_labels=xtick_labels,
# ).plot(
#     suptitle="Error Statistics vs PML Thickness",
#     output_dir=POST3, fname="pml_convergence.png",
# )
